# EMBER2024 model training and evaluation

**AAI-540 Group 1 — Gaurav SS, Marco Gonzalez Ortiz, Nitendra Tiwari**

This notebook covers the ML modeling workstream: consume the prepared Win64 data,
establish a Logistic Regression baseline, compare Logistic Regression, Random Forest,
and XGBoost, select a frozen model and threshold using validation data, and perform a
separate final temporal test. It runs locally for debugging or uses managed SageMaker
training jobs in each teammate's own AWS account.

The [project scope](../docs/modeling_scope.md), [modeling design](../docs/ml_system_design.md),
and [runbook](../docs/sagemaker_runbook.md) explain the source requirements and decisions.
Outputs are cleared for sharing. Real performance is populated only by a data-backed run.

## 1. Environment and repository

Use a Python 3.12 kernel. Install the pinned modeling requirements before imports and
restart the kernel if it has already loaded different NumPy/scikit-learn versions.
The Linux managed job installs the CPU XGBoost dependency from
`sagemaker_src/requirements.txt`, so its runtime needs package-network access or a
reviewed custom image with that dependency preinstalled.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "malware_mlops").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the cloned project branch.")
os.chdir(ROOT)
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements-sagemaker.txt"])
for path in (ROOT / "src", ROOT / "sagemaker_src"):
    sys.path.insert(0, str(path))
print("Repository:", ROOT)

Repository: C:\NitzDoc\USD\AAI_540_MLops\6-7FinalProject\AAI-540-Final-Team-Project-Group1-main\AAI-540-Final-Team-Project-Group1-main


## 2. Configure the modeling run

Copy `configs/sagemaker.example.json` to the ignored `configs/sagemaker.local.json`.
For managed training, supply your member identifier, existing private S3 bucket,
region, authorized SageMaker execution role, and the prepared handoff S3 prefix.
Use the AWS credential provider chain; do not put credentials in cells or Git.

The default switches perform no training and call no AWS APIs. Local mode uses the
same training scripts as SageMaker. Set `RUN_AWS=True` for authorized S3 transfers
or managed jobs, and choose `EXECUTION_MODE="sagemaker"` for managed training.
The model-selection weights in the example are the RFC's proposed policy. Review
the baseline before choosing `MAX_FPR` and recording a decision in section 6.

In [2]:
import json
from copy import deepcopy
from datetime import datetime, timezone
from malware_mlops.aws_workflow import (
    load_config, create_aws_context, download_handoff, validate_handoff,
    upload_training_inputs, write_selection_policy, run_baseline_training,
    run_model_comparison, collect_training_artifacts,
)

EXECUTION_MODE = "local"  # "local" or "sagemaker"
RUN_AWS = False
DOWNLOAD_HANDOFF = False
RUN_BASELINE = False
RUN_COMPARISON = False
RUN_FINAL_TEST = False
POLICY_READY = False
MAX_FPR = None
DECISION_NOTE = ""

config_path = Path(os.environ.get("AAI540_CONFIG", "configs/sagemaker.local.json"))
if not config_path.exists():
    config_path = Path("configs/sagemaker.example.json")
overrides = {"aws": {}, "data": {}}
for variable, section, key in [
    ("AWS_REGION", "aws", "region"), ("AAI540_BUCKET", "aws", "bucket"),
    ("SAGEMAKER_ROLE_ARN", "aws", "role_arn"),
    ("AAI540_HANDOFF_S3_URI", "data", "handoff_s3_uri"),
]:
    if os.environ.get(variable):
        overrides[section][key] = os.environ[variable]
if os.environ.get("AAI540_MEMBER"):
    overrides["member"] = os.environ["AAI540_MEMBER"]
cfg = load_config(config_path, overrides)
cfg["cloud_execution"] = RUN_AWS
if EXECUTION_MODE not in ("local", "sagemaker"):
    raise ValueError("Choose local or sagemaker execution mode.")
REPORT_DIR = ROOT / cfg["reports_dir"] / cfg["run_id"]
REPORT_DIR.mkdir(parents=True, exist_ok=True)
context = None
handoff = None
inputs = None
baseline_artifacts = None
comparison_artifacts = None
policy_path = None
final_report = None
print(json.dumps({"mode": EXECUTION_MODE, "member": cfg["member"], "run_id": cfg["run_id"],
                  "aws_enabled": RUN_AWS, "selection_policy_approved": cfg["selection"]["approved"]}, indent=2))

{
  "mode": "local",
  "member": "your-name",
  "run_id": "20261017T230817-ec8f12",
  "aws_enabled": false,
  "selection_policy_approved": false
}


## 3. Load the train and validation handoff

Use the data owner's matching feature schema, split manifest, and CSVs. The first
40 weeks form train; the next 12 form validation; the first six official-test weeks
form the final project test; the newest six weeks remain reserved production.
This is an **official-test-derived temporal subset**, as documented in the tracker
and repository, rather than the entire untouched official test partition.

Initially only train and validation records are opened or downloaded. Validation
checks the schema fingerprint, exact ordered columns, finite features, binary labels,
unique IDs, class counts, and chronology. The RFC's minimum of 10,000 examples per
class is checked against actual model-training data, before optional training sampling.
Use only `model_feature_columns` as inputs; labels and lineage are excluded.

> **Prerequisite & Data Status Notice:**
> This strict SageMaker-ready workflow requires the authoritative `feature_schema.json`
> (with `fit_scope="model_train_only"`) and matching splits in `data/processed/ember2024_win64_static_v1`
> or an S3 handoff bucket. If authoritative raw JSONL data has not been regenerated locally,
> the cell below will explicitly report the missing prerequisite. For a completed local
> exploratory workflow against the reconstructed approximation, see
> `notebooks/02_sagemaker_training_evaluation_local_exploratory_label_fix.ipynb`.

In [3]:
if RUN_AWS:
    context = create_aws_context(cfg)
    print(json.dumps({"identity": context["identity"], "region": context["region"],
                      "role_arn": context["role_arn"]}, indent=2))
    if DOWNLOAD_HANDOFF:
        downloaded = download_handoff(cfg, context, include_test=False)
        print("Downloaded train/validation artifacts:", sorted(downloaded))
if (ROOT / cfg["data"]["processed_dir"] / "feature_schema.json").exists():
    handoff = validate_handoff(cfg, include_test=False)
    print(json.dumps({"schema_sha256": handoff["schema_sha256"], "splits": handoff["splits"]}, indent=2))
else:
    print(f"Prerequisite missing: Authoritative handoff '{cfg['data']['processed_dir']}/feature_schema.json' was not found.")
    print("Reason: Authoritative raw EMBER2024 JSONL data is not available locally to run raw feature preparation.")
    print("Note: For exploratory modeling on the reconstructed data, use 'notebooks/02_sagemaker_training_evaluation_local_exploratory_label_fix.ipynb'.")


Prerequisite missing: Authoritative handoff 'data/processed/ember2024_win64_static_v1/feature_schema.json' was not found.
Reason: Authoritative raw EMBER2024 JSONL data is not available locally to run raw feature preparation.
Note: For exploratory modeling on the reconstructed data, use 'notebooks/02_sagemaker_training_evaluation_local_exploratory_label_fix.ipynb'.


In [4]:
if handoff is not None:
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    counts = pd.DataFrame({name: details["class_distribution"] for name, details in handoff["splits"].items()}).T
    counts = counts.rename(columns={"0": "Benign", "1": "Malicious"}).astype(int)
    counts.plot.bar(stacked=True, title="Modeling input class counts", figsize=(7, 4))
    plt.ylabel("Unique records")
    plt.tight_layout()
    plt.show()
    display(counts)
    print("Approved feature order:", handoff["schema"]["model_feature_columns"])
else:
    print("No data has been loaded.")

No data has been loaded.


## 4. Establish the Logistic Regression benchmark

The benchmark fits only training rows. Its validation scores show the operating
tradeoff at several FPR budgets and at threshold 0.5. It does not choose a final
test threshold or claim production performance. The default 250,000-row stratified
training cap controls CPU cost and is recorded; set the configured cap to zero to
use every training row. Validation remains complete.

Managed mode starts a paid SageMaker training job. Each job writes its own model
archive and report to your run prefix. Keep this configuration and run ID stable
while progressing through the notebook.

In [5]:
def local_training_command(phase, model_dir, output_dir):
    return [
        sys.executable, "sagemaker_src/train.py", "--phase", phase,
        "--train-dir", str(Path(handoff["paths"]["train"]).parent),
        "--validation-dir", str(Path(handoff["paths"]["validation"]).parent),
        "--schema-dir", str(Path(handoff["paths"]["schema"]).parent),
        "--model-dir", str(model_dir), "--output-dir", str(output_dir),
        "--seed", str(cfg["compute"]["seed"]),
        "--max-train-rows", str(cfg["compute"]["max_train_rows"]),
        "--n-jobs", str(cfg["compute"].get("n_jobs", 2)),
        "--allow-legacy-preprocessing", str(cfg["data"]["allow_legacy_preprocessing"]).lower(),
    ]

if RUN_BASELINE:
    if handoff is None:
        raise RuntimeError("Validate the train/validation handoff first.")
    if EXECUTION_MODE == "sagemaker":
        if not RUN_AWS:
            raise RuntimeError("Enable RUN_AWS before starting a managed job.")
        inputs = upload_training_inputs(cfg, context, handoff)
        baseline_estimator = run_baseline_training(cfg, context, inputs, wait=True)
        baseline_artifacts = collect_training_artifacts(cfg, context, baseline_estimator, label="baseline")
    else:
        model_dir = REPORT_DIR / "baseline" / "model"
        subprocess.check_call(local_training_command("baseline", model_dir, REPORT_DIR / "baseline" / "output"))
        baseline_artifacts = {"model_dir": str(model_dir), "report": json.loads((model_dir / "baseline_report.json").read_text())}
    print("Baseline report saved; inspect validation results before choosing a selection policy.")
else:
    print("Baseline training disabled.")

Baseline training disabled.


## 5. Inspect baseline threshold tradeoffs

The FPR budget remains a design decision after this validation-only baseline.
Small benign samples limit FPR resolution; an observed rate is not a confidence
bound or a guarantee about future traffic. Benchmark class prevalence also differs
from an operational security queue. Do not use the final test to choose the budget.

In [6]:
if baseline_artifacts is not None:
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    baseline_report = baseline_artifacts["report"]
    sweep = pd.DataFrame([{ "max_fpr": row["max_fpr"], "threshold": row["threshold"],
                            "recall": row["validation"]["recall"], "fpr": row["validation"]["fpr"],
                            "precision": row["validation"]["precision"], "f1": row["validation"]["f1"] }
                           for row in baseline_report["threshold_sweep"]])
    display(sweep)
    sweep.plot(x="fpr", y="recall", marker="o", title="Baseline validation operating points", figsize=(7, 4))
    plt.xlabel("Observed benign false positive rate")
    plt.ylabel("Malware recall")
    plt.tight_layout()
    plt.show()
    print("Baseline digest:", baseline_report["report_sha256"])
else:
    print("Run the baseline to populate this comparison.")

Run the baseline to populate this comparison.


## 6. Record the selection policy before model comparison

Set `MAX_FPR` and `DECISION_NOTE` after reviewing baseline validation results, adjust
the weights/reference values in the loaded configuration if needed, then explicitly
set `POLICY_READY=True` in the next cell. This records the choice, time, baseline
digest, and proposed weights as a frozen policy. The example's security weights
are recall 30%, FPR 25%, PR AUC 15%, F1 10%, latency 10%, size 5%, and interpretability 5%.

Latency and size use declared reference scales so smaller is better; FPR is scored
relative to the selected budget. Interpretability ratings are subjective design
priors, not measured classification quality. Precision, ROC AUC, accuracy, specificity,
balanced accuracy, and deployment complexity remain visible outside the weighted score.
The measured latency is warm local single-record CPU `predict_proba` latency; it is
not a SageMaker endpoint measurement or an SLA.

In [7]:
# Set these only after inspecting the baseline; do not choose them using test outcomes.
# MAX_FPR = 0.01
# DECISION_NOTE = "Explain the analyst false-alert budget and the baseline tradeoff."
# POLICY_READY = True
if POLICY_READY:
    if baseline_artifacts is None:
        raise RuntimeError("An actual baseline report is required before recording the policy.")
    if MAX_FPR is None or not DECISION_NOTE.strip():
        raise ValueError("Record an FPR budget and a decision rationale after baseline review.")
    selected_policy = deepcopy(cfg["selection"])
    selected_policy.update({"approved": True, "max_fpr": float(MAX_FPR),
                            "baseline_report_sha256": baseline_artifacts["report"]["report_sha256"],
                            "approved_at_utc": datetime.now(timezone.utc).isoformat(),
                            "decision_note": DECISION_NOTE.strip()})
    cfg["selection"] = selected_policy
    policy_path = write_selection_policy(cfg, REPORT_DIR / "policy" / "selection_policy.json",
                                         baseline_report=baseline_artifacts["report"])
    print(json.dumps(selected_policy, indent=2))
else:
    print("Selection policy remains unapproved; model comparison and final test are not enabled.")

Selection policy remains unapproved; model comparison and final test are not enabled.


## 7. Compare and tune the three candidate families

The bounded search fits Logistic Regression, Random Forest, and XGBoost hyperparameter
trials on train only. It selects each threshold and compares candidates on validation
under the frozen policy. No test rows are supplied to the training job. Record every
trial's parameters, validation metrics, timing, serialized model size, and weighted
score. A deterministic seed controls sampling and estimator randomness; wall-clock
latency can vary with the machine and runtime conditions.

In [8]:
if RUN_COMPARISON:
    if baseline_artifacts is None or policy_path is None:
        raise RuntimeError("Run the baseline and record the frozen selection policy first.")
    if EXECUTION_MODE == "sagemaker":
        if not RUN_AWS:
            raise RuntimeError("Enable RUN_AWS before managed comparison training.")
        if inputs is None:
            inputs = upload_training_inputs(cfg, context, handoff)
        comparison_estimator = run_model_comparison(
            cfg, context, inputs, policy_path,
            Path(baseline_artifacts["model_dir"]) / "baseline_report.json", wait=True)
        comparison_artifacts = collect_training_artifacts(cfg, context, comparison_estimator, label="compare")
    else:
        model_dir = REPORT_DIR / "comparison" / "model"
        command = local_training_command("compare", model_dir, REPORT_DIR / "comparison" / "output")
        command += ["--policy-dir", str(Path(policy_path).parent), "--baseline-dir", baseline_artifacts["model_dir"]]
        subprocess.check_call(command)
        comparison_artifacts = {"model_dir": str(model_dir), "report": json.loads((model_dir / "training_report.json").read_text())}
    print("Comparison finished; the winning model and threshold are now frozen.")
else:
    print("Candidate comparison disabled.")

Candidate comparison disabled.


## 8. Review the comparison and validation diagnostics

Compare the strongest configuration per family alongside the full trial table.
The score expresses the declared selection policy, not a universal model-quality
ranking. Inspect threshold tradeoffs, false positive counts, latency, model size,
and the qualitative deployment notes before handing the chosen artifact to another
workstream. These are validation diagnostics used for selection.

In [9]:
if comparison_artifacts is not None:
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    comparison_report = comparison_artifacts["report"]
    def trial_row(row):
        return {"trial": row["trial_id"], "family": row["name"], "recall": row["validation"]["recall"],
                "fpr": row["validation"]["fpr"], "precision": row["validation"]["precision"],
                "f1": row["validation"]["f1"], "pr_auc": row["validation"]["pr_auc"],
                "roc_auc": row["validation"]["roc_auc"], "accuracy": row["validation"]["accuracy"],
                "p95_ms": row["prediction_latency"]["p95_ms"], "size_mb": row["model_size_mb"],
                "fit_seconds": row["fit_seconds"], "weighted_score": row["weighted_score"],
                "deployment_complexity": row["deployment_complexity"]}
    trials = pd.DataFrame([trial_row(row) for row in comparison_report["candidates"]])
    families = pd.DataFrame([trial_row(row) for row in comparison_report["family_comparison"]]).set_index("family")
    display(trials)
    display(families)
    families[["recall", "pr_auc", "weighted_score"]].plot.bar(title="Validation model comparison", figsize=(9, 4))
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()
    print("Selected family:", comparison_report["selected_model"], "trial:", comparison_report["selected_trial"])
    print("Frozen threshold:", comparison_report["threshold"], "policy hash:", comparison_report["policy_sha256"])
else:
    print("No candidate results yet.")

No candidate results yet.


In [10]:
if comparison_artifacts is not None:
    import joblib
    from training import load_frame
    from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
    bundle = joblib.load(Path(comparison_artifacts["model_dir"]) / "model.joblib")
    validation_frame = load_frame(Path(handoff["paths"]["validation"]), handoff["schema"])
    validation_probabilities = bundle["estimator"].predict_proba(
        validation_frame[bundle["feature_columns"]].to_numpy())[:, 1]
    figure, axes = plt.subplots(1, 3, figsize=(16, 4))
    RocCurveDisplay.from_predictions(validation_frame["label"], validation_probabilities, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(validation_frame["label"], validation_probabilities, ax=axes[1])
    ConfusionMatrixDisplay.from_predictions(validation_frame["label"], validation_probabilities >= bundle["threshold"],
                                            labels=[0, 1], ax=axes[2], colorbar=False)
    figure.suptitle("Validation diagnostics used for model selection")
    figure.tight_layout()
    plt.show()

## 9. Run the separate final temporal test

Set `RUN_FINAL_TEST=True` only after freezing the comparison result. This stage
downloads/opens the test split for the first time and applies the saved model and
threshold without tuning or refitting. In managed mode the trained artifact is
evaluated in this SageMaker notebook kernel. A final test outcome must not be used
to adjust the same experiment and still be reported as a fresh holdout evaluation.
The production reserve is never opened by this notebook.

Use only artifacts from your own trusted training run: joblib/pickle files can
execute code when loaded. Raw executables are not required or executed.

In [11]:
if RUN_FINAL_TEST:
    if comparison_artifacts is None:
        raise RuntimeError("Freeze a model comparison result before final testing.")
    if RUN_AWS and DOWNLOAD_HANDOFF:
        download_handoff(cfg, context, include_test=True)
    final_handoff = validate_handoff(cfg, include_test=True)
    from training import file_sha256
    for split, expected_hash in comparison_artifacts["report"]["data_sha256"].items():
        if file_sha256(Path(final_handoff["paths"][split])) != expected_hash:
            raise ValueError("The train/validation handoff changed after selection; retain the original frozen experiment.")
    final_dir = REPORT_DIR / "final_test"
    subprocess.check_call([
        sys.executable, "sagemaker_src/evaluate.py", "--model-dir", comparison_artifacts["model_dir"],
        "--test-dir", str(Path(final_handoff["paths"]["test"]).parent),
        "--schema-dir", str(Path(final_handoff["paths"]["schema"]).parent),
        "--output-dir", str(final_dir), "--confirm-final-test", "true",
    ])
    final_report = json.loads((final_dir / "evaluation.json").read_text())
    print("Final test completed at the frozen threshold; retain its exact provenance and caveats.")
else:
    print("Final test remains unopened by the modeling workflow.")

Final test remains unopened by the modeling workflow.


In [12]:
if final_report is not None:
    import pandas as pd
    from IPython.display import Image, display
    display(pd.DataFrame({name: entry["value"] for name, entry in final_report["classification_metrics"].items()},
                         index=["final_temporal_test"]).T)
    print("Confusion counts:", final_report["confusion_matrix"])
    weekly = pd.DataFrame(final_report.get("weekly_metrics", {})).T
    if not weekly.empty:
        weekly[["recall", "fpr"]].plot(marker="o", title="Weekly final-test performance", figsize=(8, 4))
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()
    for path in sorted((REPORT_DIR / "final_test").glob("*.png")):
        display(Image(filename=str(path)))

## 10. Save the modeling handoff

Hand off the selected `model.joblib`, `feature_schema.json`, `training_report.json`,
the frozen selection policy, source/data hashes, runtime versions, and final-test
`evaluation.json` with its plots when available. Preserve the baseline report so
the chosen operating budget is traceable. `inference.py` documents the prepared,
schema-ordered feature request and score/label response contract for integration.

Run the repository's modeling tests and notebook checks before sharing changes.
The other workstreams can integrate these artifacts with their deployment and
operations code. Report measured local results and limitations accurately; Win64
benchmark results do not establish coverage of every malware family or future file.

In [13]:
print("Run artifacts:", REPORT_DIR)
print("Baseline available:", baseline_artifacts is not None)
print("Frozen comparison available:", comparison_artifacts is not None)
print("Final test evaluated:", final_report is not None)
print("Checks: python -m unittest discover -s tests -v")
print("Notebook checks: python scripts/validate_notebooks.py")

Run artifacts: C:\NitzDoc\USD\AAI_540_MLops\6-7FinalProject\AAI-540-Final-Team-Project-Group1-main\AAI-540-Final-Team-Project-Group1-main\reports\model_training\20261017T230817-ec8f12
Baseline available: False
Frozen comparison available: False
Final test evaluated: False
Checks: python -m unittest discover -s tests -v
Notebook checks: python scripts/validate_notebooks.py
